# Machine Learning Coursework 2 (6001CEM)
## Part C: Exploratory Data Analysis & Data Quality Investigation
**Module Code:** 6001CEM | **Credit:** 5 Credits
**Dataset:** Default of Credit Card Clients (30,000 observations)
---
### Required Tasks Covered in this Notebook:
1. **Descriptive Statistics:** Measures of central tendency, spread, and range.
2. **Correlation Analysis:** Pearson & Spearman matrices and collinearity detection.
3. **Feature Distributions & Skewness:** Distribution shape, Fisher-Pearson skewness values.
4. **Missing Value Investigation:** Null counts and implicit missing data.
5. **Duplicate Analysis:** Detailed breakdown of the 35 duplicate client pairs.
6. **Outlier Analysis:** IQR detection, boxplots across bill and payment features.
7. **Class Imbalance Investigation:** Ratio of non-defaulters to defaulters (78:22).
8. **Noise & Inconsistencies:** Irregular category codes in `EDUCATION` (0,5,6) and `MARRIAGE` (0).
9. **Critical Discussion:** How each data quality issue impacts machine learning algorithms.

### 1. Setup & Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

df = pd.read_csv('default of credit card clients.csv')
print(f"Dataset loaded: {df.shape[0]:,} observations, {df.shape[1]} attributes.")

### 2. Descriptive Statistics

In [ ]:
# Numerical summary
numeric_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3']
desc_stats = df[numeric_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats.round(2)

### 3. Correlation Analysis & Multicollinearity

In [ ]:
# Correlation heatmap among billing and repayment amounts
corr_cols = ['LIMIT_BAL', 'PAY_1', 'PAY_2', 'PAY_3', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'PAY_AMT1', 'PAY_AMT2', 'dpnm']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap (Selected Features & Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key Finding: Extreme collinearity between consecutive billing statements (BILL_AMT1 & BILL_AMT2: r = 0.95).")
print("PAY_1 (most recent repayment status) has the strongest single linear correlation with default (r = 0.32).")

### 4. Feature Distributions & Skewness Analysis

In [ ]:
# Calculate skewness for all numerical features
skew_values = df.skew().sort_values(ascending=False)
skew_df = pd.DataFrame({'Skewness': skew_values.round(3)})
print("Top 10 Most Positively Skewed Features:")
print(skew_df.head(10))

# Visualise distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['LIMIT_BAL'], kde=True, color='#1f77b4', ax=axes[0, 0], bins=30)
axes[0, 0].set_title(f"LIMIT_BAL Distribution (Skew: {df['LIMIT_BAL'].skew():.2f})", fontweight='bold')

sns.histplot(df['AGE'], kde=True, color='#2ca02c', ax=axes[0, 1], bins=30)
axes[0, 1].set_title(f"AGE Distribution (Skew: {df['AGE'].skew():.2f})", fontweight='bold')

sns.histplot(df['BILL_AMT1'], kde=True, color='#ff7f0e', ax=axes[1, 0], bins=30)
axes[1, 0].set_title(f"BILL_AMT1 Distribution (Skew: {df['BILL_AMT1'].skew():.2f})", fontweight='bold')

sns.histplot(np.log1p(df['PAY_AMT1'].clip(lower=0)), kde=True, color='#d62728', ax=axes[1, 1], bins=30)
axes[1, 1].set_title(f"Log(PAY_AMT1 + 1) Distribution (After Log Transform)", fontweight='bold')

plt.tight_layout()
plt.show()

### 5. Missing Values & Duplicate Records Analysis

In [ ]:
# Missing values
missing = df.isnull().sum()
print(f"Total explicit null/missing values: {missing.sum()}")

# Duplicate records excluding arbitrary ID
feature_cols = [c for c in df.columns if c != 'ID']
dup_count = df.duplicated(subset=feature_cols).sum()
print(f"Exact duplicate client entries found: {dup_count} (35 duplicate pairs, 70 total rows)")

# Inspect sample duplicates
dup_rows = df[df.duplicated(subset=feature_cols, keep=False)].sort_values(by=feature_cols)
print("Sample Duplicate Pair (IDs 19 and 13107):")
dup_rows[dup_rows['ID'].isin([19, 13107])][['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'BILL_AMT1', 'dpnm']]

### 6. Outlier Analysis

In [ ]:
# Outlier detection using IQR on billing and payment features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3']], palette='Blues', ax=axes[0])
axes[0].set_title('Boxplots: Bill Statement Amounts', fontweight='bold')
axes[0].set_ylabel('NT Dollars')

sns.boxplot(data=df[['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3']], palette='Greens', ax=axes[1])
axes[1].set_title('Boxplots: Payment Amounts', fontweight='bold')
axes[1].set_ylabel('NT Dollars')

plt.tight_layout()
plt.show()

print("Outlier Insight: Extreme payments and bills exist (up to NT$1M+). These represent legitimate high-net-worth accounts, not errors. Robust scaling is required rather than row deletion.")

### 7. Noise & Categorical Inconsistencies

In [ ]:
print("=== CATEGORICAL VALUE COUNTS ===")
print("EDUCATION codes:", df['EDUCATION'].value_counts().to_dict())
print("MARRIAGE codes:", df['MARRIAGE'].value_counts().to_dict())

print("\nInconsistency Identified:")
print("- EDUCATION contains undocumented values: 0 (14 rows), 5 (280 rows), 6 (51 rows).")
print("- MARRIAGE contains undocumented value: 0 (54 rows).")
print("- Solution: Consolidate undocumented EDUCATION codes into 4 ('Others'), and MARRIAGE 0 into 3 ('Others').")

### 8. Impact of Data Quality Issues on Machine Learning Models

| Data Quality Issue | Impact on Linear Models / Logistic Regression | Impact on Distance-Based Models (KNN/SVM) | Impact on Tree-Based Models (Random Forest/XGBoost) |
| :--- | :--- | :--- | :--- |
| **Multicollinearity** ($r > 0.9$ among bills) | Inflates coefficient variance, ruins interpretability, unstable parameter estimates. | Minimal impact on distance calculations. | Negligible impact; trees select features sequentially. |
| **Extreme Skewness & Outliers** | Skews linear hyperplanes; heavy leverage points degrade decision boundaries. | Completely dominates Euclidean distances; nearest neighbors distorted. | Highly robust; monotonic transformations do not change split decisions. |
| **Class Imbalance** (78:22) | Biases decision boundary toward majority class; high false negative rate. | Majority class overwhelms neighborhood voting. | Tree splits favor purity of the majority class; minority patterns missed. |
| **Duplicates** (35 pairs) | Artificially inflates train/test performance; introduces data leakage if split across folds. | Distorts local density estimates. | Overfits to duplicated feature points. |